Purpose: preprocess the UCI Air Quality dataset - clean missing sentinels, impute gaps with XGBoost models, add time features, and scale continuous variables for modeling.

Install dataset helper and XGBoost (run once if needed).

In [1]:
%pip install ucimlrepo xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Load libraries used for preprocessing.

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from ucimlrepo import fetch_ucirepo
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

Fetch the dataset and display metadata and variable info.

In [3]:
air_quality = fetch_ucirepo(id=360)
X = air_quality.data.features
y = air_quality.data.targets

print(air_quality.metadata)
print(air_quality.variables)

{'uci_id': 360, 'name': 'Air Quality', 'repository_url': 'https://archive.ics.uci.edu/dataset/360/air+quality', 'data_url': 'https://archive.ics.uci.edu/static/public/360/data.csv', 'abstract': 'Contains the responses of a gas multisensor device deployed on the field in an Italian city. Hourly responses averages are recorded along with gas concentrations references from a certified analyzer. ', 'area': 'Computer Science', 'tasks': ['Regression'], 'characteristics': ['Multivariate', 'Time-Series'], 'num_instances': 9358, 'num_features': 15, 'feature_types': ['Real'], 'demographics': [], 'target_col': None, 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2008, 'last_updated': 'Sun Mar 10 2024', 'dataset_doi': '10.24432/C59K5F', 'creators': ['Saverio Vito'], 'intro_paper': {'ID': 420, 'type': 'NATIVE', 'title': 'On field calibration of an electronic nose for benzene estimation in an urban pollution monitoring scenario', 'authors': 

Combine predictors and targets into one DataFrame.

In [4]:
df = pd.concat([X, y], axis=1)

Replace sentinel -200 with NaN and count missing values.

In [5]:
df.replace(-200, np.nan, inplace=True)
print('Missing values:')
print(df.isnull().sum())

Missing values:
Date                0
Time                0
CO(GT)           1683
PT08.S1(CO)       366
NMHC(GT)         8443
C6H6(GT)          366
PT08.S2(NMHC)     366
NOx(GT)          1639
PT08.S3(NOx)      366
NO2(GT)          1642
PT08.S4(NO2)      366
PT08.S5(O3)       366
T                 366
RH                366
AH                366
dtype: int64


Build datetime index from Date and Time, set it as index, and drop raw columns.

In [6]:
df['datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'].str.replace('.', ':'),
    format='%m/%d/%Y %H:%M:%S'
)
df.set_index('datetime', inplace=True)
df.drop(['Date', 'Time'], axis=1, inplace=True)

Add calendar features for grouping and downstream models.

In [7]:
df['hour'] = df.index.hour
df['weekday'] = df.index.weekday
df['month'] = df.index.month

Preview cleaned data and shape.

In [8]:
display(df.head())
print(df.shape)

,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,hour,weekday,month
datetime,,,,,,,,,,,,,,,,
2004-03-10 18:00:00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578,18,2,3
2004-03-10 19:00:00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255,19,2,3
2004-03-10 20:00:00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502,20,2,3
2004-03-10 21:00:00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867,21,2,3
2004-03-10 22:00:00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888,22,2,3


(9357, 16)


Define imputation target groups and predictor subsets.

In [9]:
small_cols = [
    'PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH', 'C6H6(GT)'
]
medium_cols = ['CO(GT)', 'NOx(GT)', 'NO2(GT)']
big_col = 'NMHC(GT)'

predictor_subsets = {
    'NOx(GT)': ['PT08.S1(CO)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'CO(GT)', 'NO2(GT)', 'hour', 'RH'],
    'PT08.S3(NOx)': ['PT08.S1(CO)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'CO(GT)', 'NO2(GT)', 'NOx(GT)'],
    'PT08.S4(NO2)': ['PT08.S1(CO)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'PT08.S5(O3)', 'CO(GT)', 'T', 'RH', 'AH'],
    'NMHC(GT)': ['PT08.S1(CO)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'CO(GT)', 'NOx(GT)', 'NO2(GT)', 'hour', 'weekday', 'month'],
}


Hyperparameter grid and helper to optionally tune XGBoost imputers.

In [10]:
param_grid = {
    'max_depth': [3, 4, 5],
    'min_child_weight': [2, 3, 5],
    'subsample': [0.6, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.8, 0.9],
    'learning_rate': [0.01, 0.05, 0.09],
    'n_estimators': [200, 500, 1000],
}

def fill_missing_tuned(df, target_col, feature_cols, n_iter=10):
    """Run RandomizedSearchCV XGBoost to impute missing target_col."""
    train_df = df[df[target_col].notna()]
    missing_df = df[df[target_col].isna()]

    if len(missing_df) == 0:
        print(f'No missing values for {target_col}.')
        return df

    X_train = train_df[feature_cols]
    y_train = train_df[target_col]

    xgb = XGBRegressor(objective='reg:squarederror', random_state=42)
    search = RandomizedSearchCV(
        estimator=xgb,
        param_distributions=param_grid,
        n_iter=n_iter,
        scoring='r2',
        cv=3,
        verbose=1,
        n_jobs=-1,
        random_state=43,
    )
    search.fit(X_train, y_train)

    best_model = search.best_estimator_
    print(f'{target_col} best params: {search.best_params_}')
    print(f'{target_col} best CV R^2: {search.best_score_:.3f}')

    X_missing = missing_df[feature_cols]
    df.loc[df[target_col].isna(), target_col] = best_model.predict(X_missing)
    return df

Optional: rerun hyperparameter search (disabled by default to keep runs quick).

In [11]:
run_search = False
if run_search:
    for col in small_cols:
        features = predictor_subsets.get(col, [c for c in df.columns if c != col])
        df = fill_missing_tuned(df, col, features, n_iter=100)

    for col in medium_cols:
        features = predictor_subsets.get(col, [c for c in df.columns if c != col])
        df = fill_missing_tuned(df, col, features, n_iter=100)

    features = predictor_subsets.get(big_col, [c for c in df.columns if c != big_col])
    df = fill_missing_tuned(df, big_col, features, n_iter=100)

    display(df.head())

Helper to impute using stored best parameters.

In [12]:
def fill_missing_with_best(df, target_col, feature_cols, best_params):
    train_df = df[df[target_col].notna()]
    missing_df = df[df[target_col].isna()]

    if len(missing_df) == 0:
        print(f'No missing values for {target_col}.')
        return df

    X_train = train_df[feature_cols]
    y_train = train_df[target_col]

    model = XGBRegressor(objective='reg:squarederror', random_state=42, **best_params)
    model.fit(X_train, y_train)

    X_missing = missing_df[feature_cols]
    df.loc[df[target_col].isna(), target_col] = model.predict(X_missing)

    return df

Stored best hyperparameters and CV R^2 scores from prior search.

In [13]:
best_params_dict = {
    'PT08.S1(CO)': {'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.6},  # R^2: 0.834
    'PT08.S2(NMHC)': {'subsample': 0.7, 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.07, 'colsample_bytree': 0.9},  # R^2: 0.998
    'PT08.S3(NOx)': {'subsample': 0.9, 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 0.6},  # R^2: 0.524
    'PT08.S4(NO2)': {'subsample': 0.6, 'n_estimators': 400, 'min_child_weight': 3, 'max_depth': 4, 'learning_rate': 0.05, 'colsample_bytree': 0.6},  # R^2: 0.484
    'PT08.S5(O3)': {'subsample': 0.7, 'n_estimators': 600, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.07, 'colsample_bytree': 0.7},  # R^2: 0.899
    'T': {'subsample': 0.6, 'n_estimators': 600, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.07, 'colsample_bytree': 0.9},  # R^2: 0.948
    'RH': {'subsample': 0.6, 'n_estimators': 600, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.07, 'colsample_bytree': 0.9},  # R^2: 0.962
    'AH': {'subsample': 0.9, 'n_estimators': 600, 'min_child_weight': 4, 'max_depth': 5, 'learning_rate': 0.07, 'colsample_bytree': 0.8},  # R^2: 0.899
    'C6H6(GT)': {'subsample': 0.6, 'n_estimators': 600, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.07, 'colsample_bytree': 0.9},  # R^2: 0.997
    'CO(GT)': {'subsample': 0.8, 'n_estimators': 1000, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.05, 'colsample_bytree': 0.6},  # R^2: 0.858
    'NOx(GT)': {'subsample': 0.7, 'n_estimators': 600, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.05, 'colsample_bytree': 0.6},  # R^2: 0.511
    'NO2(GT)': {'subsample': 0.8, 'n_estimators': 1000, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 0.9},  # R^2: 0.745
    'NMHC(GT)': {'subsample': 0.6, 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 4, 'learning_rate': 0.01, 'colsample_bytree': 0.9},  # R^2: 0.572
}

Impute missing values using the stored best parameters.

In [14]:
for col in small_cols:
    features = predictor_subsets.get(col, [c for c in df.columns if c != col])
    df = fill_missing_with_best(df, col, features, best_params_dict[col])

for col in medium_cols:
    features = predictor_subsets.get(col, [c for c in df.columns if c != col])
    df = fill_missing_with_best(df, col, features, best_params_dict[col])

features = predictor_subsets.get(big_col, [c for c in df.columns if c != big_col])
df = fill_missing_with_best(df, big_col, features, best_params_dict[big_col])

print('Remaining missing values after imputation:')
print(df.isnull().sum())
print(df.shape)

Remaining missing values after imputation:
CO(GT)           0
PT08.S1(CO)      0
NMHC(GT)         0
C6H6(GT)         0
PT08.S2(NMHC)    0
NOx(GT)          0
PT08.S3(NOx)     0
NO2(GT)          0
PT08.S4(NO2)     0
PT08.S5(O3)      0
T                0
RH               0
AH               0
hour             0
weekday          0
month            0
dtype: int64
(9357, 16)


Scale continuous features (fit on training split in real pipelines).

In [15]:
continuous_features = ['PT08.S1(CO)','PT08.S2(NMHC)','PT08.S3(NOx)','PT08.S4(NO2)','PT08.S5(O3)', 'CO(GT)','NOx(GT)','NO2(GT)','C6H6(GT)','NMHC(GT)', 'T', 'RH', 'AH']
scaler = StandardScaler()

df[continuous_features] = scaler.fit_transform(df[continuous_features])

Save the preprocessed dataset to CSV for reuse.

In [16]:
df.to_csv('preprocessed_air_quality.csv', index=True)

Preview preprocessed data.

In [17]:
display(df.head())
print(df.shape)

,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,hour,weekday,month
datetime,,,,,,,,,,,,,,,,
2004-03-10 18:00:00,0.345514,1.029801,-0.402597,0.057638,0.221365,-0.354188,0.898330,0.065624,0.487077,0.513683,-0.555160,-0.093845,-0.701403,18,2,3
2004-03-10 19:00:00,-0.073785,0.739028,-0.630636,-0.202266,-0.064850,-0.663469,1.357292,-0.391267,0.152876,-0.193473,-0.589759,-0.159346,-0.776599,19,2,3
2004-03-10 20:00:00,0.065981,1.209396,-0.774660,-0.243850,-0.115173,-0.526011,1.225048,0.087381,0.142824,0.050209,-0.751217,0.184534,-0.719096,20,2,3
2004-03-10 21:00:00,0.065981,1.098218,-0.822668,-0.223058,-0.086866,-0.324732,1.038352,0.261435,0.215696,0.358395,-0.855011,0.512040,-0.634123,21,2,3
2004-03-10 22:00:00,-0.353318,0.653506,-0.996697,-0.503753,-0.439130,-0.526011,1.477866,0.130894,-0.020507,0.136215,-0.831946,0.490206,-0.629234,22,2,3


(9357, 16)
